# Lição 6 · O gerador nasce

As cinco peças juntas: doze despachos do Word, conferidos contra o painel antes de confiar. É o **Alvo** do curso.

**Cenário fictício.** Na Comarca de Guaraí, a **Dalva** é assessora do cartório que atende as três varas. Toda semana ela prepara os despachos padrão da pauta. A comarca descrita, as pessoas e os processos são inventados para o curso; os números de processo têm dígito verificador propositalmente inválido. No Tocantins todo processo é digital e tramita no eproc.

**Como usar:** execute as células de cima para baixo (▶ ou Ctrl+Enter). O Colab amanhece limpo: ao reabrir, use *Ambiente de execução → Executar tudo*. As células marcadas **Sua vez** são para você escrever; a célula **Conferir** logo abaixo diz se está certo.

**Arquivos:** `pauta.xlsx` (12 processos) e `modelo_despacho.docx`, um modelo **fictício** com lacunas `{{ }}`
criado para o curso; para o ofício, `peritos.xlsx` e `modelo_oficio.docx`. A célula abaixo baixa tudo.
Os documentos gerados são minutas para a Dalva conferir antes de levar ao eproc.

A docxtpl escreve arquivos do Word e não vem instalada no Colab. A célula instala só se precisar.

In [1]:
try:
    import docxtpl
    print("docxtpl já instalada")
except ImportError:
    %pip install -q docxtpl

docxtpl já instalada


In [2]:
# Baixa as planilhas e os modelos do material de apoio, só se ainda não estiverem nesta pasta.
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/rogernogueira/curso_python/main/"
for caminho in [
    "dados/pauta.xlsx",
    "modelos/modelo_despacho.docx",
    "dados/peritos.xlsx",
    "modelos/modelo_oficio.docx",
]:
    nome = os.path.basename(caminho)
    if os.path.exists(nome):
        print("já está aqui:", nome)
    else:
        urllib.request.urlretrieve(BASE + caminho, nome)
        print("baixado:", nome)

já está aqui: pauta.xlsx
já está aqui: modelo_despacho.docx
já está aqui: peritos.xlsx
já está aqui: modelo_oficio.docx


## 1. As ferramentas e a data de referência

`import os` e `import shutil` trazem ferramentas de pastas e arquivos; `from datetime import date` traz a ferramenta de datas,
e `date(2026, 9, 15)` cria uma data. O caderno usa `date.today()`; o curso fixa a data de referência, para os resultados
não mudarem com o calendário. O nome em maiúsculas, `HOJE`, é a convenção para um valor que não muda.

In [3]:
import os
import shutil
import pandas as pd
from datetime import date
from docxtpl import DocxTemplate

# Data de referência do curso, no lugar de date.today()
HOJE = date(2026, 9, 15)

## 2. As peças: a decisão e a preparação

`texto_decisao` é a das lições 2 a 4 (versão corrigida; o caderno usa o `else` que arquiva). `preparar` é a síntese do curso:
limpa o nome, normaliza o status, formata o valor e escolhe a frase. Ela faz o papel da `montar_despacho` das lições 3 e 4;
a diferença é que aqui o formato fica no modelo do Word, não no código.

In [4]:
def texto_decisao(status):
    if status == "concluso":
        return "Autos conclusos. Decido."
    elif status == "prazo vencido":
        return "Certifique-se o decurso do prazo e voltem conclusos."
    elif status == "aguardando":
        return "Aguarde-se o cumprimento. Após, conclusos."
    elif status == "arquivado":
        return "Arquivem-se os autos."
    else:
        return "Situação não prevista: verificar."

In [5]:
def preparar(processo):
    return {
        "numero_processo": processo["numero_processo"],
        "nome_parte": str(processo["nome_parte"]).strip().title(),
        "vara": processo["vara"],
        "valor_causa": f"R$ {processo['valor_causa']:.2f}",
        "texto_decisao": texto_decisao(str(processo["status"]).strip().lower()),
        "data_hoje": HOJE.strftime("%d/%m/%Y"),
    }

## 3. O gerador

`shutil.rmtree("saida", ignore_errors=True)` apaga a pasta de saída antes de cada rodada; `os.makedirs` a recria vazia.
Sem isso, os arquivos de uma rodada anterior continuam lá e a contagem não bate (o `exist_ok=True` não esvazia a pasta).
No laço, `iterrows()` entrega as linhas; o `_` descarta o índice; `render` preenche as lacunas; `save` grava o arquivo.

In [6]:
import os
import shutil
import pandas as pd
from datetime import date
from docxtpl import DocxTemplate

# Data de referência do curso, no lugar de date.today()
HOJE = date(2026, 9, 15)

def texto_decisao(status):
    if status == "concluso":
        return "Autos conclusos. Decido."
    elif status == "prazo vencido":
        return "Certifique-se o decurso do prazo e voltem conclusos."
    elif status == "aguardando":
        return "Aguarde-se o cumprimento. Após, conclusos."
    elif status == "arquivado":
        return "Arquivem-se os autos."
    else:
        return "Situação não prevista: verificar."

def preparar(processo):
    return {
        "numero_processo": processo["numero_processo"],
        "nome_parte": str(processo["nome_parte"]).strip().title(),
        "vara": processo["vara"],
        "valor_causa": f"R$ {processo['valor_causa']:.2f}",
        "texto_decisao": texto_decisao(str(processo["status"]).strip().lower()),
        "data_hoje": HOJE.strftime("%d/%m/%Y"),
    }

shutil.rmtree("saida", ignore_errors=True)
os.makedirs("saida", exist_ok=True)
df = pd.read_excel("pauta.xlsx")

for _, linha in df.iterrows():
    dados = preparar(linha.to_dict())
    doc = DocxTemplate("modelo_despacho.docx")
    doc.render(dados)
    nome_limpo = dados["numero_processo"].replace(".", "").replace("-", "")
    doc.save("saida/despacho_" + nome_limpo + ".docx")

print("Despachos gerados:", len(df))
print("Arquivos na pasta saida:", len(os.listdir("saida")))

Despachos gerados: 12
Arquivos na pasta saida: 12


Abra um despacho gerado e confira: sempre confira.

In [7]:
from docx import Document

# O despacho da Ana, o primeiro processo da pauta
nome_limpo = df.iloc[0]["numero_processo"].replace(".", "").replace("-", "")
for paragrafo in Document("saida/despacho_" + nome_limpo + ".docx").paragraphs:
    if paragrafo.text.strip():
        print(paragrafo.text)

COMARCA DE GUARAÍ · 1ª Vara Cível
Processo nº 0004512-33.2026.8.27.0001
Requerente: Ana Ribeiro Nogueira
Valor da causa: R$ 15400.00
DESPACHO
Autos conclusos. Decido.
Guaraí, 15/09/2026.
Modelo fictício criado para o curso; não é documento oficial.


O aviso `Despachos gerados: len(df)` conta **linhas da planilha**, não arquivos. Quem confere é `len(os.listdir("saida"))`.

## 4. O requisito novo: só os vencidos

O filtro da lição 5, com `hoje` em `pd.Timestamp`, porque a coluna do pandas não se compara com `date`.
O processo de controle é o painel: 5 vencidos na pauta.

In [8]:
def gerar(linhas):
    shutil.rmtree("saida", ignore_errors=True)
    os.makedirs("saida", exist_ok=True)
    for _, linha in linhas.iterrows():
        dados = preparar(linha.to_dict())
        doc = DocxTemplate("modelo_despacho.docx")
        doc.render(dados)
        nome_limpo = dados["numero_processo"].replace(".", "").replace("-", "")
        doc.save("saida/despacho_" + nome_limpo + ".docx")
    return len(os.listdir("saida"))

hoje = pd.Timestamp("2026-09-15")
print("Arquivos com o filtro de vencidos:", gerar(df[df["data_prazo"] < hoje]))

Arquivos com o filtro de vencidos: 5


## 5. A IA redige a minuta, você assina

Três códigos "da IA" para o mesmo pedido. Rodar sem erro não é estar certo: confira cada um contra o controle (5).

In [9]:
# Código 1 da IA (ia1)
vencidos = df[df["data_prazo"] > hoje]
print("ia1:", gerar(vencidos), "arquivos na pasta saida")

ia1: 7 arquivos na pasta saida


In [10]:
# Código 2 da IA (ia2)
vencidos = df[df["data_prazo"].astype(str) < "15/09/2026"]
print("ia2:", gerar(vencidos), "arquivos na pasta saida")

ia2: 0 arquivos na pasta saida


In [11]:
# Código 3 da IA (ia3)
vencidos = df[df["data_prazo"] < hoje]
print("ia3:", gerar(vencidos), "arquivos na pasta saida")

ia3: 5 arquivos na pasta saida


### Sua vez: aprove ou reprove

Para cada código, escreva `"aprovado"` ou `"reprovado"`, pelos números.

In [12]:
# Sua vez: escreva aqui
veredito = {"ia1": ..., "ia2": ..., "ia3": ...}

In [13]:
# Conferir (não precisa editar)
certos = {"ia1": "reprovado", "ia2": "reprovado", "ia3": "aprovado"}
motivos = {
    "ia1": "inverte a comparação: gera os 7 que NÃO venceram",
    "ia2": "compara data com texto: não gera nenhum arquivo, sem erro",
    "ia3": "dá 5, como o painel",
}
for nome, certo in certos.items():
    dado = veredito.get(nome, ...)
    if dado is ...:
        print(f"✕ {nome}: escreva aprovado ou reprovado.")
    else:
        print(("✓ " if dado == certo else "✕ ") + f"{nome}: {certo} ({motivos[nome]}).")

✕ ia1: escreva aprovado ou reprovado.
✕ ia2: escreva aprovado ou reprovado.
✕ ia3: escreva aprovado ou reprovado.


## 6. Transferência: o ofício circular aos peritos

O mesmo gerador, outro documento. O número do ofício, `041/2026`, tem uma barra, que o Python leria como pasta;
por isso a barra vira traço no nome do arquivo.

In [14]:
def preparar_oficio(perito):
    return {
        "numero_oficio": perito["numero_oficio"],
        "nome_perito": perito["nome_perito"],
        "especialidade": perito["especialidade"],
        "data_hoje": HOJE.strftime("%d/%m/%Y"),
    }

In [15]:
shutil.rmtree("saida", ignore_errors=True)
os.makedirs("saida", exist_ok=True)
peritos = pd.read_excel("peritos.xlsx")

for _, linha in peritos.iterrows():
    dados = preparar_oficio(linha.to_dict())
    doc = DocxTemplate("modelo_oficio.docx")
    doc.render(dados)
    doc.save("saida/oficio_" + dados["numero_oficio"].replace("/", "-") + ".docx")

print("Ofícios na pasta saida:", len(os.listdir("saida")))

Ofícios na pasta saida: 6


## Para conferir com a lição

- Esvazie a pasta `saida` antes de cada rodada e conte os arquivos, não as linhas.
- Toda saída nova passa pelo processo de controle (o número do painel) antes de ser aceita.
- A régua de qualidade: funciona do zero, aguenta troca de dados, você entende cada linha e é útil de verdade.

**Fim do curso.** Para levar à sua rotina: troque a planilha pela sua, rode o painel; ajuste os nomes de colunas
(`df.columns`); troque o modelo; por fim, ajuste a `preparar`. Um passo de cada vez, executando a cada mudança.